# cloud

> Run a notebook or a script as a job, in the background here or on Colab, Kaggle or SageMaker, on the hardware you pick; follow its progress beside your other jobs; and bring its model back to this machine.

In [ ]:
#| default_exp cloud

In [ ]:
#| export
import base64, datetime, io, json, os, re, shutil, signal, subprocess, sys, tarfile, tempfile, time, uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path

import torch
from fastcore.basics import patch

import sysone
from sysone.core import dispatch
from sysone.record import git_info

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne, test_stdout
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
from sysone.inference import Decider
import nbformat
from mlflow import MlflowClient
from sysone.track import tracking_uri

A training run that takes hours belongs somewhere other than the notebook that wrote it: a Colab or Kaggle GPU, a SageMaker training job, or at least a background process on this machine. `remote` runs a notebook or a script there, as it is:

```python
job = remote("train.ipynb", on="colab", hw="t4", data="data/emotions.jsonl", push="me/emotions-lora", track=True)
job.wait()                        # progress lines as they come: step, loss, speed, time left, memory
jobs()                            # every job, on every backend, in one table
exp = job.fetch()                 # the export, back here
dec = Decider.load(exp)           # its answers
learn = load_learner(exp, data)   # or its learner, to evaluate, interpret or train on here
```

Three kinds of things describe a job, each a small type:

| What | Types |
|---|---|
| where it runs | `Local()`, `Colab()`, `Kaggle()`, `SageMaker()` |
| on what | `CPU()`, `MPS()`, `GPU("T4")`, `GPU("T4", 2)`, `TPU("v5e-1")`, `Neuron("trn1.2xlarge")` |
| on which data | a local path, `hf:owner/name` (a Hub dataset), `kaggle:owner/name`, `s3://bucket/key` |

and what a backend does with them are [generic functions](00_core.ipynb#generic-functions), dispatched on the pairs:

- `machine(backend, hardware)` is how the backend names the hardware (`--gpu L4` for Colab, `NvidiaL4X1` for Kaggle, `ml.g6.xlarge` for SageMaker), and an error, not a silent substitute, when it has none;
- `stage(data, backend, name)` is how the data reaches the job (shipped with it, a Kaggle dataset, an S3 channel, or downloaded from the Hub by the job);
- `submit`, `poll`, `cancel` and `collect` start, follow, end and bring back a job.

This notebook defines them, the job's files and the [Local](#local-jobs) backend; [colab](45_colab.ipynb), [kaggle](46_kaggle.ipynb) and [aws](48_aws.ipynb) add their methods.

A job's life:

```{mermaid}
sequenceDiagram
    participant N as your notebook
    participant C as sysone.cloud
    participant B as the backend's machine
    participant H as Hugging Face Hub
    N->>C: remote("train.ipynb", on=..., hw=..., data=..., push=...)
    C->>C: the job's folder: a wheel of this sysone, the entry, small data, job.json, the runner
    C->>B: submit: the folder, on the machine for the hardware
    B->>B: the runner installs sysone, runs the entry, prints SYSONE progress lines
    N->>C: job.wait(), jobs()
    C->>B: poll: new log lines, status
    B->>H: the runner pushes the export (with push=...)
    N->>C: job.fetch()
    C->>N: the outputs: export, executed notebook, MLflow runs, log
```

## Hardware

In [ ]:
#| export
@dataclass(frozen=True)
class Hardware:
    "What a job runs on"
    def __str__(self): return type(self).__name__.lower()

@dataclass(frozen=True)
class CPU(Hardware):
    "A CPU machine; `high_mem` for the larger memory shape where a backend has one"
    high_mem:bool = False

@dataclass(frozen=True)
class MPS(Hardware):
    "Apple's GPU, on a Mac"

GPU_NAMES = ("T4", "L4", "G4", "A10G", "A100", "H100", "RTX6000")

@dataclass(frozen=True)
class GPU(Hardware):
    "`count` NVIDIA GPUs of model `name`: T4, L4, G4, A10G, A100, H100 or RTX6000 (the RTX PRO 6000)"
    name:str = "T4"
    count:int = 1
    high_mem:bool = False
    def __post_init__(self):
        object.__setattr__(self, "name", self.name.upper())
        if self.name not in GPU_NAMES: raise ValueError(f"unknown GPU {self.name!r}; one of {GPU_NAMES}")
    def __str__(self): return self.name + (f"x{self.count}" if self.count > 1 else "")

@dataclass(frozen=True)
class TPU(Hardware):
    "A Google TPU slice: v5e-1 or v6e-1 (Colab), v5e-8 or v6e-8 (Kaggle)"
    kind:str = "v5e-1"
    def __str__(self): return f"tpu {self.kind}"

@dataclass(frozen=True)
class Neuron(Hardware):
    "AWS Trainium or Inferentia, named by instance type: trn1.2xlarge, trn2.48xlarge, inf2.xlarge, ..."
    kind:str = "trn1.2xlarge"
    def __str__(self): return f"neuron {self.kind}"

def hardware(spec) -> Hardware:
    "Hardware from a short name: cpu, mps, t4, t4x2, l4, a100, h100, tpu, v5e-1, tpu:v6e-8, trn1.2xlarge, inf2.xlarge"
    if isinstance(spec, Hardware): return spec
    s = str(spec).strip().lower()
    if s in ("cpu", "cpu-highmem"): return CPU(high_mem=s.endswith("highmem"))
    if s == "mps": return MPS()
    if s.startswith("tpu"): return TPU(s.split(":", 1)[1] if ":" in s else "v5e-1")
    if re.fullmatch(r"v\d+e-\d+", s): return TPU(s)
    if s.startswith(("trn", "inf", "neuron")): return Neuron(s.split(":", 1)[1] if ":" in s else s)
    m = re.fullmatch(r"(?:gpu:)?([a-z0-9]+?)(?:x(\d+))?", s)
    if m and m.group(1).upper() in GPU_NAMES: return GPU(m.group(1), int(m.group(2) or 1))
    raise ValueError(f"unknown hardware {spec!r}; try cpu, mps, t4, t4x2, l4, a100, h100, tpu, v5e-1 or trn1.2xlarge")

In [ ]:
test_eq([hardware(s) for s in ("cpu", "mps", "t4x2", "L4", "a100", "tpu", "v6e-8", "trn1.2xlarge")],
        [CPU(), MPS(), GPU("T4", 2), GPU("L4"), GPU("A100"), TPU("v5e-1"), TPU("v6e-8"), Neuron("trn1.2xlarge")])
test_eq(str(GPU("t4", 2)), "T4x2")
test_fail(lambda: hardware("k80"), contains="unknown hardware")
test_fail(lambda: GPU("V100"), contains="unknown GPU")

Each piece of hardware has the learner preset that suits it (see [learner](05_learner.ipynb#presets)): the job sets `SYSONE_PRESET`, so the same notebook trains with the right batch size and precision wherever it runs.

In [ ]:
#| export
@dispatch
def preset_for(hw:CPU) -> str: return "cpu"
@dispatch
def preset_for(hw:MPS) -> str: return "macbook"
@dispatch
def preset_for(hw:GPU) -> str:
    if hw.name == "T4": return "t4x2" if hw.count > 1 else "t4"
    return "l4" if hw.name in ("L4", "G4", "A10G") else "a100"
@dispatch
def preset_for(hw:TPU) -> str: return "tpu"
@dispatch
def preset_for(hw:Neuron) -> str: return "neuron"

In [ ]:
test_eq([preset_for(h) for h in (CPU(), MPS(), GPU("T4"), GPU("T4", 2), GPU("L4"), GPU("H100"))], ["cpu", "macbook", "t4", "t4x2", "l4", "a100"])

## Backends

A backend is where a job runs, with its own settings. `Local` is this machine; the others hold what their service needs (see their notebooks).

In [ ]:
#| export
@dataclass
class Backend:
    "Where a job runs"
    @property
    def name(self) -> str: return type(self).__name__.lower()

@dataclass
class Local(Backend):
    "A background process on this machine, in this environment; at most `max_jobs` at once"
    max_jobs:int = 1

@dataclass
class Colab(Backend):
    "A Colab VM through the `colab` CLI; `secrets` names the environment variables passed to it (the Hub token, for pushing)"
    auth:str = "oauth2"
    keep:bool = False
    secrets:tuple = ("HF_TOKEN",)

@dataclass
class Kaggle(Backend):
    "A Kaggle kernel through the `kaggle` CLI, under `owner` (the logged-in account by default)"
    owner:str|None = None
    private:bool = True
    internet:bool = True

@dataclass
class SageMaker(Backend):
    "A SageMaker training job; `endpoint_url` points at an emulator such as floci, `local=True` uses the SDK's local mode"
    role:str|None = None
    bucket:str|None = None
    region:str|None = None
    endpoint_url:str|None = None
    image:str|None = None
    local:bool = False

BACKENDS = {"local": Local, "colab": Colab, "kaggle": Kaggle, "sagemaker": SageMaker}
BACKEND_MODULES = {"colab": "sysone.colab", "kaggle": "sysone.kaggle", "sagemaker": "sysone.aws"}

def backend(spec, **kw) -> Backend:
    "A backend from its name (`local`, `colab`, `kaggle`, `sagemaker`) and settings, or the backend itself"
    if isinstance(spec, Backend): return spec
    if str(spec).lower() not in BACKENDS: raise ValueError(f"unknown backend {spec!r}; one of {sorted(BACKENDS)}")
    return BACKENDS[str(spec).lower()](**kw)

def load_backend(b:Backend) -> bool:
    "Import the module that holds a backend's methods; True if it was not loaded yet"
    mod = BACKEND_MODULES.get(b.name)
    if mod is None or mod in sys.modules: return False
    import importlib
    importlib.import_module(mod)
    return True

`machine` is how a backend names a piece of hardware; a backend that does not have it says so. The method on `(Backend, Hardware)` is the fallback: it loads the backend's module, whose methods are then tried, and fails if none fits.

In [ ]:
#| export
@dispatch
def machine(b:Backend, hw:Hardware):
    "How backend `b` names hardware `hw`: an error when it does not have it"
    if load_backend(b): return machine(b, hw)
    raise ValueError(f"{b.name} has no {hw}")

@dispatch
def machine(b:Local, hw:CPU) -> str: return "cpu"

@dispatch
def machine(b:Local, hw:MPS) -> str:
    if not (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()): raise ValueError("this machine has no MPS device")
    return "mps"

@dispatch
def machine(b:Local, hw:GPU) -> str:
    n = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if n < hw.count: raise ValueError(f"this machine has {n} CUDA devices, not {hw.count}")
    return f"cuda x{hw.count}"

def default_hardware(b:Backend) -> Hardware:
    "What a job runs on when no hardware is named: this machine's MPS or CUDA device, and a CPU elsewhere"
    if isinstance(b, Local):
        if torch.cuda.is_available(): return GPU(next((n for n in GPU_NAMES if n in torch.cuda.get_device_name(0).upper()), "A100"))
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available(): return MPS()
    return CPU()

In [ ]:
test_eq(machine(Local(), CPU()), "cpu")
test_fail(lambda: machine(Local(), TPU()), contains="local has no tpu v5e-1")
test_eq(backend("kaggle", owner="me"), Kaggle(owner="me"))

## Data

Data is named by where it lives: a local path, `hf:owner/name[@revision]` for a Hub dataset, `kaggle:owner/name` for a Kaggle dataset, `s3://bucket/key` for S3. A job finds its data in `SYSONE_DATA` (and `SYSONE_DATA_VALID` and so on, for data passed by name), as a local path on the job's machine: `job_input` reads it, and falls back to a path of your choice when the notebook runs here, so one notebook runs unchanged in both places.

In [ ]:
#| export
@dataclass(frozen=True)
class Data:
    "Where a job's data lives"

@dataclass(frozen=True)
class LocalPath(Data):
    "A file or folder on this machine"
    path:str

@dataclass(frozen=True)
class HubData(Data):
    "A Hub dataset repo, at a revision"
    repo:str
    revision:str|None = None

@dataclass(frozen=True)
class KaggleData(Data):
    "A Kaggle dataset, `owner/name`"
    ref:str

@dataclass(frozen=True)
class S3Data(Data):
    "An S3 prefix"
    uri:str

def as_data(x) -> Data:
    "Data from a local path, `hf:owner/name[@revision]`, `kaggle:owner/name` or `s3://bucket/key`"
    if isinstance(x, Data): return x
    s = str(x)
    if s.startswith("hf:"):
        repo, _, rev = s[3:].partition("@")
        return HubData(repo, rev or None)
    if s.startswith("kaggle:"): return KaggleData(s[len("kaggle:"):])
    if s.startswith("s3://"): return S3Data(s)
    if Path(s).exists(): return LocalPath(str(Path(s).resolve()))
    raise ValueError(f"{s!r} is no local path; name remote data hf:owner/name, kaggle:owner/name or s3://bucket/key")

def data_var(name:str="data") -> str:
    "The environment variable that holds a job's data called `name`"
    return "SYSONE_DATA" if name == "data" else f"SYSONE_DATA_{re.sub(r'[^A-Za-z0-9]', '_', name).upper()}"

def job_input(name:str="data", default=None) -> Path|None:
    "Where a job's data called `name` is on the job's machine, or `default` outside a job"
    v = os.environ.get(data_var(name))
    return Path(v) if v else (Path(default) if default is not None else None)

def in_job() -> bool:
    "Whether this runs inside a sysone job"
    return bool(os.environ.get("SYSONE_JOB"))

def job_output(default=None) -> Path:
    "Where a job's entry should export its model (`SYSONE_OUTPUT`), or `default` outside a job"
    return Path(os.environ.get("SYSONE_OUTPUT") or default or tempfile.mkdtemp())

In [ ]:
test_eq((as_data("hf:me/emotions@v2"), as_data("kaggle:me/emotions"), as_data("s3://b/k")),
        (HubData("me/emotions", "v2"), KaggleData("me/emotions"), S3Data("s3://b/k")))
test_eq(as_data("fixtures/typed_decisions_tiny.jsonl"), LocalPath(str(Path("fixtures/typed_decisions_tiny.jsonl").resolve())))
test_fail(lambda: as_data("me/emotions"), contains="no local path")
test_eq((data_var(), data_var("valid")), ("SYSONE_DATA", "SYSONE_DATA_VALID"))
test_eq(job_input("data", "x.jsonl"), Path("x.jsonl"))

`stage(data, backend, var)` says how data reaches a job: environment variables for it, files to ship with it, and the backend's own mounts (a Kaggle dataset, an S3 channel). Hub data reaches every backend the same way: the job downloads it (a private repo needs the Hub token among the job's secrets). Local data reaches a local job by its path.

In [ ]:
#| export
@dispatch
def stage(d:Data, b:Backend, var:str) -> dict:
    "How data `d` reaches a job on `b` as variable `var`: `env` for the job, `files` to ship, `hub` to download, and the backend's mounts"
    if load_backend(b): return stage(d, b, var)
    raise ValueError(f"{b.name} cannot read {type(d).__name__} data")

@dispatch
def stage(d:HubData, b:Backend, var:str) -> dict: return {"hub": {var: d.repo + (f"@{d.revision}" if d.revision else "")}}

@dispatch
def stage(d:LocalPath, b:Local, var:str) -> dict: return {"env": {var: d.path}}

In [ ]:
test_eq(stage(HubData("me/x", "v1"), Local(), "SYSONE_DATA"), {"hub": {"SYSONE_DATA": "me/x@v1"}})
test_eq(stage(LocalPath("/d/x.jsonl"), Local(), "SYSONE_DATA"), {"env": {"SYSONE_DATA": "/d/x.jsonl"}})

## Jobs

A `Job` records what runs (the entry), where (backend, machine) and what became of it (status, outputs, the last progress line). Jobs are kept in `~/.config/sysone/jobs.json` (or `SYSONE_JOBS`), and their folders under `~/.cache/sysone/jobs` (or `SYSONE_JOBS_DIR`), so a later session, or `sysone jobs`, finds them again.

In [ ]:
#| export
DONE = ("complete", "error", "timeout", "stopped")

@dataclass
class Job:
    "A notebook or script run as a job"
    id:str
    backend:str
    machine:str
    entry:str
    options:dict = field(default_factory=dict)   # the backend's settings
    hardware:str = ""
    ref:str|None = None                          # the backend's handle: a process id, a Colab session, a Kaggle kernel, a training job
    folder:str|None = None                       # the job's files on this machine
    status:str = "created"
    submitted:float|None = None
    ended:float|None = None
    push:str|None = None
    url:str|None = None
    outputs:str|None = None
    last:dict|None = None                        # the last progress line
    offset:int = 0                               # how much of the log has been read
    @property
    def done(self) -> bool: return self.status in DONE
    @property
    def backend_obj(self) -> Backend: return backend(self.backend, **self.options)

def jobs_file() -> Path: return Path(os.environ.get("SYSONE_JOBS") or Path.home() / ".config" / "sysone" / "jobs.json")
def jobs_dir() -> Path: return Path(os.environ.get("SYSONE_JOBS_DIR") or Path.home() / ".cache" / "sysone" / "jobs")

def load_jobs() -> list:
    "Every recorded job"
    f = jobs_file()
    if not f.exists(): return []
    keep = set(Job.__dataclass_fields__)
    return [Job(**{k: v for k, v in j.items() if k in keep}) for j in json.loads(f.read_text())]

def save_job(job:Job) -> Job:
    "Record a job (replacing its earlier record)"
    jobs = [j for j in load_jobs() if j.id != job.id] + [job]
    jobs_file().parent.mkdir(parents=True, exist_ok=True)
    tmp = jobs_file().with_suffix(".tmp")
    tmp.write_text(json.dumps([asdict(j) for j in jobs], indent=1, default=str)); tmp.replace(jobs_file())
    return job

def get_job(id:str) -> Job:
    "A recorded job by id (or a unique prefix of it)"
    hits = [j for j in load_jobs() if j.id == id] or [j for j in load_jobs() if j.id.startswith(id)]
    if len(hits) != 1: raise KeyError(f"{len(hits)} jobs match {id!r}")
    return hits[0]

In [ ]:
os.environ["SYSONE_JOBS"] = str(Path(tempfile.mkdtemp()) / "jobs.json")
os.environ["SYSONE_JOBS_DIR"] = tempfile.mkdtemp()
save_job(Job("local-1", "local", "cpu", "train.py"))
save_job(Job("local-1", "local", "cpu", "train.py", status="complete"))
test_eq([(j.id, j.status, j.done) for j in load_jobs()], [("local-1", "complete", True)])
test_eq(get_job("local").id, "local-1")

## The job's files

A job ships as one folder: the entry (notebook or script), its small data, `job.json` (what to run and how), a wheel of the sysone this machine runs, and the runner. The wheel matters: a job runs the code you have here, uncommitted changes and all, not whatever release is on PyPI (from an installed release, the job installs that release). The [training record](05a_record.ipynb) of a model trained remotely therefore names the same commit as your checkout.

In [ ]:
#| export
def source_root() -> Path|None:
    "The checkout this process's sysone comes from (a folder with pyproject.toml), or None for an installed release"
    p = Path(sysone.__file__).resolve().parent.parent
    return p if (p / "pyproject.toml").exists() and (p / "sysone").is_dir() else None

def build_wheel(dest) -> Path|None:
    "A wheel of this process's sysone checkout in `dest`, or None for an installed release"
    root = source_root()
    if root is None: return None
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    cmd = (["uv", "build", "--wheel", "--out-dir", str(dest), str(root)] if shutil.which("uv")
           else [sys.executable, "-m", "pip", "wheel", "--no-deps", "-w", str(dest), str(root)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"building sysone's wheel failed: {(r.stdout + r.stderr)[-1000:]}")
    return max(dest.glob("sysone-*.whl"), key=os.path.getmtime)

The runner is one file that runs on the job's machine with nothing but Python. It unpacks the job (Kaggle uploads a single file, so there the folder travels inside the runner), installs the wheel over the machine's own torch and transformers (Colab's and Kaggle's already satisfy sysone), downloads Hub data, and runs the entry with `SYSONE_JOB`, `SYSONE_PRESET`, `SYSONE_PROGRESS` and `SYSONE_OUTPUT` set, so the learner reports and exports by itself, and with `SYSONE_SOURCE`, the git state of the checkout the job was sent from, which the training record falls back on. A notebook runs cell by cell with nbdev's own runner (cells marked `#| eval: false` skipped) and comes back executed. At the end the runner keeps the export, pushes it to the Hub when asked, writes `status.json`, and packs everything in `outputs.tar.gz` (a pushed export stays on the Hub, which `fetch` reads it from). A time limit stops the entry and marks the job `timeout`. Secrets, such as the Hub token for the push, reach the runner as a file it reads and deletes at once: never on a command line, and never in the code a backend's tool records.

In [ ]:
#| export
RUNNER = r'''# Written by sysone.cloud: runs one job. It unpacks the job's files (embedded below when the backend uploads one file),
# installs sysone over the machine's own torch, runs the notebook or script, prints SYSONE progress lines, keeps the outputs.
import base64, glob, io, json, os, shutil, signal, subprocess, sys, tarfile, time
HERE = os.path.dirname(os.path.abspath(__file__))
BUNDLE = ""

def say(event, **kw): print("SYSONE " + json.dumps(dict(t=round(time.time(), 1), event=event, **kw)), flush=True)

NB = r"""import re, sys
from fastcore.nbio import read_nb, write_nb
from execnb.shell import CaptureShell
nb = read_nb(sys.argv[1])
skip = lambda c: c.cell_type != "code" or bool(re.search(r"^#\|\s*eval:\s*false", c.source, re.M))
sh, ok = CaptureShell(sys.argv[1]), True
try: sh.run_all(nb, exc_stop=True, preproc=skip)
except BaseException as e:
    ok = False
    print(sh.prettytb(fname=sys.argv[1]) or repr(e), file=sys.stderr)
write_nb(nb, sys.argv[2])
sys.exit(0 if ok else 1)
"""

def main():
    os.makedirs(HERE, exist_ok=True)
    if BUNDLE:
        with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE))) as t: t.extractall(HERE, filter="data")
    spec = json.load(open(os.path.join(HERE, "job.json")))
    sec = os.path.join(HERE, ".secrets.json")        # secrets travel as a file, read once and deleted, never in a command or a log
    if os.path.isfile(sec):
        with open(sec) as f: os.environ.update(json.load(f))
        os.remove(sec)
    out = os.path.join(HERE, "outputs"); os.makedirs(out, exist_ok=True)
    say("job-start", job=spec["id"], backend=spec["backend"], machine=spec["machine"])
    result, t0 = {"status": "error", "returncode": None}, time.time()
    try:
        if spec.get("install"):
            pkgs = [os.path.join(HERE, p) if p.endswith(".whl") else p for p in spec["install"]]
            if shutil.which("uv"): subprocess.run(["uv", "pip", "install", "--python", sys.executable, "-q"] + pkgs, check=True)
            else:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
                whl = [p for p in pkgs if p.endswith(".whl")]      # pip keeps an installed sysone of the same version (uv does not): replace it
                if whl: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps"] + whl, check=True)
            say("installed", packages=spec["install"])
        env = dict(os.environ, **(spec.get("env") or {}))
        env.update(SYSONE_JOB=json.dumps({"backend": spec["backend"], "job": spec["id"], "machine": spec["machine"]}),
                   SYSONE_PROGRESS="1", SYSONE_OUTPUT=os.path.join(out, "export"), SYSONE_ENTRY=os.path.join(HERE, spec["entry"]))
        if spec.get("preset"): env["SYSONE_PRESET"] = spec["preset"]
        if spec.get("code"): env["SYSONE_SOURCE"] = json.dumps(spec["code"])      # the checkout the job was sent from
        if spec.get("track") is not None:
            env["SYSONE_TRACK"] = json.dumps(spec["track"])
            env["MLFLOW_TRACKING_URI"] = spec.get("tracking_uri") or "sqlite:///" + os.path.join(out, "mlflow.db")
        for var, ref in (spec.get("hub") or {}).items():
            from huggingface_hub import snapshot_download
            repo, _, rev = ref.partition("@")
            env[var] = snapshot_download(repo, repo_type="dataset", revision=rev or None, local_dir=os.path.join(HERE, "data", var.lower()))
            say("data", var=var, repo=repo)
        for var, rel in (spec.get("files") or {}).items(): env[var] = os.path.join(HERE, rel)
        for var, pattern in (spec.get("find") or {}).items():       # inputs the backend mounts (a Kaggle dataset): found by pattern
            hits = sorted(glob.glob(pattern, recursive=True))
            if not hits: raise FileNotFoundError(f"no input matches {pattern}")
            env[var] = hits[0]
        entry = os.path.join(HERE, spec["entry"])
        if entry.endswith(".ipynb"): cmd = [sys.executable, "-c", NB, entry, os.path.join(out, os.path.basename(entry))]
        elif int(spec.get("gpus") or 1) > 1:
            cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(spec["gpus"]), entry] + list(spec.get("args") or [])
        else: cmd = [sys.executable, entry] + list(spec.get("args") or [])
        say("run", entry=spec["entry"])
        p = subprocess.Popen(cmd, cwd=HERE, env=env, start_new_session=True)
        try: rc = p.wait(timeout=spec.get("max_seconds"))
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGTERM)
            try: p.wait(timeout=30)
            except subprocess.TimeoutExpired: os.killpg(p.pid, signal.SIGKILL)
            rc, result["status"] = None, "timeout"
        result["returncode"] = rc
        exp = os.path.join(out, "export")
        if not os.path.isfile(os.path.join(exp, "sysone.json")):    # an export written elsewhere: the newest folder with a sysone.json
            found = [os.path.dirname(f) for f in glob.glob(os.path.join(HERE, "**", "sysone.json"), recursive=True) if not f.startswith(out)]
            if found: shutil.copytree(max(found, key=os.path.getmtime), exp, dirs_exist_ok=True)
        if rc == 0:
            result["status"] = "complete"
            if spec.get("push") and os.path.isfile(os.path.join(exp, "sysone.json")):
                from sysone.hub import push_to_hub
                res = push_to_hub(exp, spec["push"], private=True)
                result["url"] = res.get("url")
                say("pushed", repo=spec["push"], url=res.get("url"))
    except Exception as e:
        result["error"] = repr(e)[:2000]
        say("job-error", error=repr(e)[:500])
    result["seconds"] = round(time.time() - t0, 1)
    with open(os.path.join(out, "status.json"), "w") as f: json.dump(result, f)
    pushed = os.path.join(out, "export") if result.get("url") else None     # a pushed export is on the Hub: the tarball stays small
    part = os.path.join(HERE, "outputs.tar.gz.part")            # renamed when whole: a poll may look at any moment
    with tarfile.open(part, "w:gz") as t:
        t.add(out, arcname="outputs", filter=lambda ti: None if pushed and ti.name.startswith("outputs/export") else ti)
    os.replace(part, os.path.join(HERE, "outputs.tar.gz"))
    say("job-end", **result)
    return 0 if result["status"] == "complete" else 1

if __name__ == "__main__": sys.exit(main())
'''

def runner_source(here:str|None=None, bundle:bytes|None=None) -> str:
    "The runner, optionally with a fixed working folder (`here`) and the job's files embedded (a .tar.gz)"
    src = RUNNER
    if here is not None: src = src.replace("HERE = os.path.dirname(os.path.abspath(__file__))", f"HERE = {here!r}", 1)
    if bundle is not None: src = src.replace('BUNDLE = ""', f'BUNDLE = "{base64.b64encode(bundle).decode()}"', 1)
    return src

def pack(folder, skip:tuple=("sysone_job.py",)) -> bytes:
    "A job's folder as a .tar.gz, in memory"
    folder, buf = Path(folder), io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as t:
        for p in sorted(folder.rglob("*")):
            if p.is_file() and p.name not in skip: t.add(p, arcname=str(p.relative_to(folder)))
    return buf.getvalue()

::: {.callout-note title="Finding: pip keeps an installed sysone of the same version"}
A SageMaker image has sysone in it already, the build the image was made from. Installing a job's wheel over it, pip finds the same version and keeps it, so the job would have run the image's code rather than the checkout's. uv replaces it, since it compares the files. This was checked with two builds of one version of a small package: after pip, the first build's code ran; after uv, the second's. With pip, the runner then reinstalls the wheel alone (`--force-reinstall --no-deps`).
:::

In [ ]:
compile(RUNNER, "runner", "exec")
src = runner_source("/kaggle/working/job", pack(Path("fixtures/tiny-modernbert")))
assert "HERE = '/kaggle/working/job'" in src and 'BUNDLE = "H4sI' in src
compile(src, "runner", "exec");

## Generic functions of a job

`submit` starts a job and fills in its `ref`; `poll` returns the log lines written since the last poll and updates the job's status; `cancel` ends it; `collect` brings its outputs to a folder here. A backend's notebook adds a method of each; a job's own methods (`job.wait()`, `job.stop()`, `job.fetch()`) call them.

In [ ]:
#| export
@dispatch
def submit(b:Backend, job:Job) -> Job:
    "Start `job` on `b`"
    if load_backend(b): return submit(b, job)
    raise NotImplementedError(f"{b.name} cannot run jobs")

@dispatch
def poll(b:Backend, job:Job) -> str:
    "The log text `job` wrote since the last poll; updates its status"
    if load_backend(b): return poll(b, job)
    raise NotImplementedError(f"{b.name} cannot poll jobs")

@dispatch
def cancel(b:Backend, job:Job) -> Job:
    "End `job`"
    if load_backend(b): return cancel(b, job)
    raise NotImplementedError(f"{b.name} cannot cancel jobs")

@dispatch
def collect(b:Backend, job:Job, dest:Path) -> Path:
    "Bring `job`'s outputs into `dest`; returns the outputs folder"
    if load_backend(b): return collect(b, job, dest)
    raise NotImplementedError(f"{b.name} cannot collect jobs")

def read_from(path, offset:int) -> tuple:
    "The text of a file from byte `offset` on, and the new offset"
    path = Path(path)
    if not path.exists(): return "", offset
    with open(path, "rb") as f:
        f.seek(offset); data = f.read()
    return data.decode("utf-8", errors="replace"), offset + len(data)

def unpack_outputs(tarball, dest) -> Path:
    "Unpack a job's outputs.tar.gz into `dest`; returns `dest/outputs`"
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tarball) as t: t.extractall(dest, filter="data")
    return dest / "outputs"

## remote

`remote` makes the job: it checks that the backend has the hardware, stages the data, writes the folder (with the wheel, for a remote backend), and submits it. `install=False` installs nothing on the machine (it already has what the job needs), and `packages` adds packages to install. `data` is one source or a dict of them by name; `push` names the Hub repo for the export; `track=True` (or a dict of `learn.track` options) logs the training to MLflow, in the job's own SQLite file that `fetch` brings back; `max_hours` stops a job that runs long; `env` sets variables, and `args` passes arguments to a script.

In [ ]:
#| export
def job_id(b:Backend) -> str:
    "A new job's id: backend, time and a random tag"
    return f"{b.name}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:4]}"

def remote(entry, on="local", hw=None, data=None, args:list|None=None, env:dict|None=None, push:str|None=None, track=None,
           max_hours:float|None=None, packages:list|None=None, install:bool=True, name:str|None=None, dry_run:bool=False, **backend_kw) -> Job:
    "Run a notebook or script as a job on backend `on` and hardware `hw`; returns the submitted job"
    b = backend(on, **backend_kw)
    hw = hardware(hw) if hw is not None else default_hardware(b)
    mach = machine(b, hw)
    entry = Path(entry)
    if not entry.is_file(): raise FileNotFoundError(f"no such notebook or script: {entry}")
    jid = name or job_id(b)
    folder = jobs_dir() / b.name / jid; folder.mkdir(parents=True, exist_ok=False)
    shutil.copy2(entry, folder / entry.name)
    sources = data if isinstance(data, dict) else ({"data": data} if data is not None else {})
    staged = {"env": {}, "files": {}, "hub": {}}
    for nm, d in sources.items():
        for k, v in stage(as_data(d), b, data_var(nm)).items():
            if isinstance(v, dict): staged.setdefault(k, {}).update(v)
            else: staged.setdefault(k, []).extend(v)
    for var, src in staged.get("ship", {}).items():           # files shipped with the job, under data/
        dst = folder / "data" / Path(src).name; dst.parent.mkdir(exist_ok=True)
        (shutil.copytree if Path(src).is_dir() else shutil.copy2)(src, dst)
        staged["files"][var] = str(dst.relative_to(folder))
    pkgs = None
    if install and not isinstance(b, Local):
        whl = build_wheel(folder)
        pkgs = ([whl.name] if whl is not None else [f"sysone=={sysone.__version__}"]) + list(packages or []) + (["execnb"] if entry.suffix == ".ipynb" else []) \
               + (["mlflow>=3.16"] if track else [])
    spec = {"id": jid, "backend": b.name, "machine": str(mach), "hardware": str(hw), "preset": preset_for(hw), "entry": entry.name,
            "args": list(args or []), "env": {**(env or {}), **staged["env"]}, "files": staged["files"], "hub": staged["hub"],
            "install": pkgs, "push": push, "track": (track if isinstance(track, dict) else {}) if track else None,
            "max_seconds": int(max_hours * 3600) if max_hours else None, "gpus": getattr(hw, "count", 1),
            "code": {"git": git_info(entry.parent), "sysone_git": git_info(Path(sysone.__file__).parent)}}     # for the training record
    spec |= {k: v for k, v in staged.items() if k not in ("env", "files", "hub", "ship")}
    (folder / "job.json").write_text(json.dumps(spec, indent=1))
    (folder / "sysone_job.py").write_text(runner_source())
    opts = {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(b).items()}
    job = Job(jid, b.name, str(mach), entry.name, options=opts, hardware=str(hw), folder=str(folder), push=push)
    if dry_run: job.status = "dry-run"; return job
    job = submit(b, job)
    job.submitted = job.submitted or time.time()
    return save_job(job)

## Local jobs

A local job is the runner in a background process of its own (in a new session, so it outlives the notebook that started it), writing to `job.log` in the job's folder; it runs in this environment, so nothing is installed. On a Mac it gets the MPS memory caps unless they are set already, and `Local(max_jobs=1)` keeps two trainings from sharing the GPU's memory.

In [ ]:
#| export
_PROCS = {}

def _alive(pid) -> bool:
    try: os.kill(int(pid), 0); return True
    except (OSError, ValueError, TypeError): return False

@submit.register
def submit_local(b:Local, job:Job) -> Job:
    running = [j for j in load_jobs() if j.backend == "local" and not j.done and j.status != "dry-run" and _alive(j.ref)]
    if len(running) >= b.max_jobs: raise RuntimeError(f"{len(running)} local job(s) running already: {[j.id for j in running]}; Local(max_jobs=...) allows more")
    folder, env = Path(job.folder), dict(os.environ)
    if sys.platform == "darwin": env.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.7"); env.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.6")
    with open(folder / "job.log", "ab") as log:
        p = subprocess.Popen([sys.executable, str(folder / "sysone_job.py")], stdout=log, stderr=subprocess.STDOUT, cwd=folder,
                             env=env, start_new_session=True)
    _PROCS[job.id] = p
    job.ref, job.status, job.submitted = str(p.pid), "running", time.time()
    return job

@poll.register
def poll_local(b:Local, job:Job) -> str:
    folder = Path(job.folder)
    text, job.offset = read_from(folder / "job.log", job.offset)
    st = folder / "outputs" / "status.json"
    p = _PROCS.get(job.id)
    finished = (p.poll() is not None) if p is not None else not _alive(job.ref)
    if st.exists() and finished:
        s = json.loads(st.read_text()); job.status, job.url = s["status"], s.get("url") or job.url
    elif finished and not st.exists(): job.status = "error"
    return text

@cancel.register
def cancel_local(b:Local, job:Job) -> Job:
    if not job.done and _alive(job.ref):
        try: os.killpg(int(job.ref), signal.SIGTERM)
        except ProcessLookupError: pass
    job.status = "stopped"
    return job

@collect.register
def collect_local(b:Local, job:Job, dest:Path) -> Path:
    src = Path(job.folder) / "outputs"
    if Path(dest).resolve() == src.parent.resolve(): return src
    return unpack_outputs(Path(job.folder) / "outputs.tar.gz", dest)

## Following and fetching

`job.refresh()` polls the backend once: new log text is scanned for progress lines, and the last one is kept. `job.wait()` polls until the job ends, printing each progress line in words; `job.logs()` is the whole log; `job.fetch()` brings the outputs here (into `dest`, by default the job's folder) and returns the export, or the outputs folder when there is no export; a job that should have pushed its export but could not (Kaggle has no way to pass it the Hub token) is pushed from here. With `sync=True`, the MLflow runs a job logged go into your own store (see [track](44_track.ipynb)). `load_learner(export, data)` turns an export back into a learner here, to evaluate, interpret or train on (see [inference](06_inference.ipynb#load_learner)).

In [ ]:
#| export
def parse_lines(text:str) -> list:
    "The SYSONE progress records in some log text"
    out = []
    for line in text.splitlines():
        i = line.find("SYSONE {")
        if i < 0: continue
        try: out.append(json.loads(line[i + len("SYSONE "):]))
        except json.JSONDecodeError: pass
    return out

def describe(r:dict) -> str:
    "A progress record in words"
    ev = r.get("event")
    mem = next((f"{k.split('_')[0].upper()} {r[k]:.2f} GB" for k in ("gpu_mem_gb", "mps_mem_gb", "xla_mem_gb") if isinstance(r.get(k), (int, float))), "")
    if ev == "log":
        parts = [f"step {r.get('step')}/{r.get('max_steps')}", f"loss {r['loss']:.4f}" if isinstance(r.get("loss"), (int, float)) else None,
                 f"{r['steps_per_s']:.2f} steps/s" if r.get("steps_per_s") else None, f"{r['eta_s']} s left" if r.get("eta_s") is not None else None,
                 mem or None, f"GPU {r['gpu_util']}%" if r.get("gpu_util") is not None else None]
        return " · ".join(p for p in parts if p)
    if ev == "eval": return "validation: " + ", ".join(f"{k} {v:.4g}" for k, v in r.items() if k in ("loss", "accuracy", "macro_f1", "ece") and isinstance(v, (int, float)))
    if ev == "vm": return f"machine: GPU {r.get('gpu_util')}% busy, {r.get('gpu_mem_gb')} of {r.get('gpu_total_gb')} GB"
    keep = {k: v for k, v in r.items() if k not in ("t", "event")}
    return f"{ev}: " + ", ".join(f"{k} {v}" for k, v in keep.items())

@patch
def refresh(self:Job) -> list:
    "Poll the job once; returns the new progress records"
    if self.done or self.status in ("dry-run", "created"): return []
    new = parse_lines(poll(self.backend_obj, self))
    if new: self.last = next((r for r in reversed(new) if r.get("event") == "log"), self.last)
    if self.done and self.ended is None: self.ended = time.time()
    save_job(self)
    return new

@patch
def wait(self:Job, every:float=30, logs:bool=True, timeout:float|None=None, retries:int=5) -> Job:
    "Poll until the job ends, printing its progress; a failed poll (a network blip, a busy VM) is retried, up to `retries` in a row"
    t0, failed = time.time(), 0
    while True:
        try: new, failed = self.refresh(), 0
        except Exception as e:
            failed += 1
            if failed > retries: raise
            print(f"[{self.id}] poll failed ({failed}/{retries}): {str(e)[:300]}", flush=True)
            new = []
        for r in new:
            if logs: print(f"[{self.id}] {describe(r)}", flush=True)
        if self.done or (timeout is not None and time.time() - t0 > timeout): return self
        time.sleep(every)

@patch
def logs(self:Job) -> str:
    "The job's log text so far (a remote job's, as far as polling has brought it)"
    p = Path(self.folder or "") / "job.log"
    return p.read_text(errors="replace") if p.exists() else ""

@patch
def progress(self:Job) -> list:
    "Every progress record of the job so far (from its local log; remote jobs: what polling has brought)"
    p = Path(self.folder or "") / "job.log"
    return parse_lines(p.read_text(errors="replace")) if p.exists() else []

@patch
def stop(self:Job) -> Job:
    "End the job"
    cancel(self.backend_obj, self)
    self.ended = time.time()
    return save_job(self)

@patch
def fetch(self:Job, dest=None, sync:bool=False) -> Path:
    "Bring the job's outputs here; returns its export (or the outputs folder); `sync=True` copies its MLflow runs into your store"
    out = collect(self.backend_obj, self, Path(dest or self.folder))
    self.outputs = str(out); save_job(self)
    if sync and (out / "mlflow.db").exists():
        from sysone.track import sync_run
        sync_run(f"sqlite:///{out / 'mlflow.db'}")
    if (out / "export" / "sysone.json").exists():
        if self.push and not self.url and self.status == "complete":     # the job could not push (no token there): push from here
            from sysone.hub import push_to_hub
            self.url = push_to_hub(out / "export", self.push)["url"]; save_job(self)
        return out / "export"
    if self.push and self.status == "complete":       # pushed, so left out of the outputs: read it from the Hub
        from huggingface_hub import snapshot_download
        return Path(snapshot_download(self.push, local_dir=out / "export"))
    return out

In [ ]:
fj, calls = Job("flaky", "local", "cpu", "train.py", status="running"), []
def flaky():                                       # a backend whose first poll fails
    calls.append(1)
    if len(calls) == 1: raise ConnectionError("a network blip")
    fj.status = "complete"; return []
fj.refresh = flaky
test_stdout(lambda: fj.wait(every=0), "[flaky] poll failed (1/5): a network blip")
test_eq((fj.status, len(calls)), ("complete", 2))
def down(): raise ConnectionError("the backend is down")
fj.status, fj.refresh = "running", down
test_fail(lambda: fj.wait(every=0, retries=2), contains="the backend is down")     # down for good: the error, after the retries

`jobs()` lists every job, refreshed, as a table; `watch()` keeps refreshing it until the jobs it follows end. The table shows each job's latest progress line, so several trainings on several backends read side by side.

In [ ]:
#| export
class JobList(list):
    "Jobs, shown as a table"
    def _rows(self):
        now = time.time()
        for j in self:
            r = j.last or {}
            el = ((j.ended or now) - j.submitted) if j.submitted else None
            yield [j.id, j.backend, j.hardware or j.machine, j.entry, j.status, f"{el / 60:.1f} min" if el is not None else "",
                   f"{r.get('step')}/{r.get('max_steps')}" if r else "", f"{r['loss']:.4f}" if isinstance(r.get("loss"), (int, float)) else "",
                   describe(r).split(" · ", 3)[-1] if r else ""]
    HEAD = ["job", "backend", "hardware", "entry", "status", "elapsed", "step", "loss", "last"]
    def __repr__(self):
        rows = [self.HEAD] + [[str(x) for x in r] for r in self._rows()]
        w = [max(len(r[i]) for r in rows) for i in range(len(self.HEAD))]
        return "\n".join("  ".join(c.ljust(w[i]) for i, c in enumerate(r)) for r in rows)
    def _repr_html_(self):
        th = "".join(f"<th style='text-align:left'>{h}</th>" for h in self.HEAD)
        tr = "".join("<tr>" + "".join(f"<td style='text-align:left'>{c}</td>" for c in r) + "</tr>" for r in self._rows())
        return f"<table><tr>{th}</tr>{tr}</table>"

def jobs(refresh:bool=True, backend:str|None=None, active:bool=False) -> JobList:
    "Every recorded job (of one backend, or the ones still running), refreshed"
    js = [j for j in load_jobs() if (backend is None or j.backend == backend) and (not active or not j.done)]
    if refresh:
        for j in js:
            try: j.refresh()
            except Exception as e: j.last = (j.last or {}) | {"event": "poll-error", "error": str(e)[:100]}
    return JobList(js)

def watch(every:float=30, ids:list|None=None):
    "Refresh the table of running jobs until they end: in place in a notebook, printed in a terminal"
    ids = ids or [j.id for j in load_jobs() if not j.done and j.status != "dry-run"]
    try:
        from IPython.display import display
        h = display(JobList(), display_id=True) if "ipykernel" in sys.modules else None
    except ImportError: h = None
    while True:
        js = JobList([j for j in jobs() if j.id in ids])
        if h is not None: h.update(js)
        else: print(repr(js), "\n", flush=True)
        if all(j.done for j in js): return js
        time.sleep(every)

## A local job, end to end

A script that trains the tiny model for one epoch and exports it where the job says, the same file that would run on Colab; the job runs it in the background, and the export comes back:

In [ ]:
script = Path(tempfile.mkdtemp()) / "train.py"
script.write_text("""import os
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, os.environ["ENCODER"], train="head", logging_steps=1, disable_tqdm=True)
learn.fit(2, lr=1e-3)
learn.export(job_output())
""")
fx = str(Path("fixtures/tiny-modernbert").resolve())
job = remote(script, hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": fx})
job.wait(every=1)
test_eq(job.status, "complete")

In [ ]:
recs = job.progress()
test_eq([r["event"] for r in recs][0], "job-start")
assert [r for r in recs if r["event"] == "log"] and recs[-1]["event"] == "job-end"
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
exp = job.fetch()
ans = Decider.load(exp, device="cpu").predict(rec["state"], rec["questions"])
test_eq(set(ans), set(rec["questions"]))
meta = json.loads((exp / "sysone.json").read_text())
test_eq((meta["training"]["hardware"]["run"]["backend"], meta["training"]["hardware"]["run"]["job"], meta["training"]["training"]["preset"]), ("local", job.id, "cpu"))
test_eq(json.loads((Path(job.folder) / "job.json").read_text())["code"]["sysone_git"]["commit"], git_info()["commit"])     # what the record falls back on
jobs()

A notebook runs the same way, cell by cell, and comes back executed; here with MLflow tracking, whose runs `fetch(sync=True)` copies into the local store, and a cell marked `#| eval: false` that the job skips:

In [ ]:
nbp = Path(tempfile.mkdtemp()) / "train.ipynb"
nb = nbformat.v4.new_notebook()
nb.cells = [nbformat.v4.new_code_cell(c) for c in (
    "import os\nfrom sysone.data import TypedDecisions\nfrom sysone.learner import decision_learner\nfrom sysone.cloud import job_input, job_output",
    "#| eval: false\nraise RuntimeError('a cell for interactive use only')",
    "data = TypedDecisions.from_json(job_input('data'), valid=0.25, calib=0.2)\nlearn = decision_learner(data, os.environ['ENCODER'], train='head', logging_steps=1, disable_tqdm=True)",
    "learn.fit(1, lr=1e-3)\nlearn.export(job_output())\nprint('exported')")]
nbformat.write(nb, nbp)
os.environ["SYSONE_MLFLOW_DIR"] = tempfile.mkdtemp()
njob = remote(nbp, hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": fx}, track={"experiment": "jobs", "system_metrics": False})
njob.wait(every=1, logs=False)
test_eq(njob.status, "complete")
exp2 = njob.fetch(sync=True)
ran = nbformat.read(Path(njob.outputs) / "train.ipynb", as_version=4)
test_eq("exported" in "".join(o.get("text", "") for o in ran.cells[-1].outputs), True)
runs = MlflowClient(tracking_uri()).search_runs([MlflowClient(tracking_uri()).get_experiment_by_name("jobs").experiment_id])
test_eq(runs[0].data.tags["sysone.job"], njob.id)

A job that runs past `max_hours` is stopped and marked `timeout`, and a failing entry ends as `error` with its traceback in the log:

In [ ]:
slow = Path(tempfile.mkdtemp()) / "slow.py"; slow.write_text("import time; time.sleep(60)\n")
tj = remote(slow, hw="cpu", max_hours=3 / 3600)
tj.wait(every=1, logs=False)
test_eq(tj.status, "timeout")
bad = Path(tempfile.mkdtemp()) / "bad.py"; bad.write_text("raise ValueError('no data here')\n")
bj = remote(bad, hw="cpu")
bj.wait(every=1, logs=False)
test_eq(bj.status, "error")
assert "no data here" in bj.logs()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()